# 01 · Scrape The Current's playlist history

Grabs every play from **2005-12-22** (first broadcast) through yesterday from
`https://www.thecurrent.org/playlist/the-current/YYYY-MM-DD`. One request = one full day.

- **Resumable:** each day is saved to `data/raw/YYYY/YYYY-MM-DD.csv`, and days already saved are skipped. Stop at any time and rerun.
- **Polite:** 5s (+0–1s random) between requests. It stops by itself on a 403 or 3 failed days in a row.
- **Full run ≈ 12 hours.** For the full run, the terminal is more reliable than this notebook:
  `caffeinate -i .venv/bin/python scraper.py` (see README). Use this notebook for small ranges and to inspect the data.

In [ ]:
from datetime import date, timedelta
from scraper import scrape_range, retry_failed, combine, START_DATE

## Config

In [ ]:
START = START_DATE                          # date(2005, 12, 22)
END = date.today() - timedelta(days=1)     # yesterday
DELAY = 5                                  # seconds between requests

## Scrape
Interrupt the kernel (■) any time; rerunning this cell picks up where it left off.

In [ ]:
scrape_range(START, END, delay=DELAY)

## Retry any failed days

In [ ]:
retry_failed(delay=DELAY)

## Combine into `data/plays.parquet`

In [ ]:
df = combine()
df.head()

## Sanity checks

In [ ]:
df.info()

In [ ]:
# plays per year (2005 is only 10 days; the current year is partial)
df.groupby(df.played_at.dt.year).size()

In [ ]:
# share of rows with each field filled in
df.notna().mean().round(3)

In [ ]:
# days with suspiciously few plays (possible gaps in the station's data)
per_day = df.groupby(df.played_at.dt.date).size()
per_day[per_day < 150].sort_values()